# Lennard-Jones molecular dynamics

This notebook is part of the computational resources for the Statistical Physics course at École Polytechnique. To return to the main repository, follow this link: [https://github.com/cossio/StatPhysCompX](https://github.com/cossio/StatPhysCompX).

In this notebook, we will simulate a simple system of particles interacting via a Lennard-Jones potential. We will calculate the radial distribution function and observe the system's different phases (solid, liquid, gas).

This script will:
- Initialize the positions and velocities of N particles under a Lennard-Jones potential.
- Run a molecular dynamics simulation using the Velocity Verlet algorithm.
- Show an animation of the particle trajectories.
- Calculate the radial distribution function from the final particle positions and plot it.

The behavior of the system depends heavily on its density ($\rho$) and temperature ($T_0$). You can edit these parameters to simulate different phases of matter:
- **Liquid Phase** (as configured): With $\rho = 0.8$ and $T_0 = 1.0$, the system should behave like a liquid. The RDF will show a strong peak at the nearest-neighbor distance ($r \approx \sigma$) followed by decaying oscillations.
- **Gas Phase**: Try a low density and high temperature (e.g., $\rho = 0.2$, $T_0 = 2.0$). The particles will move more freely, and the RDF will be close to 1 for $r > \sigma$.
- **Solid Phase**: Try a high density and low temperature (e.g., $\rho = 1.2$, $T_0 = 0.5$). The particles should arrange into a crystal lattice.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
N = 64            # Number of particles
rho = 0.8         # Density
T0 = 1.0          # Target temperature
L_box = (N / rho)**(1/3)  # Box length from density
dt = 0.005        # Time step
n_steps = 2000    # Number of simulation steps
eps = 1.0         # LJ energy parameter
sigma_lj = 1.0    # LJ length parameter
r_cut = 2.5 * sigma_lj  # Cutoff radius
r_cut2 = r_cut**2
mass = 1.0        # Particle mass

In [ ]:
def init_positions(N, L_box):
    """Place particles on a cubic lattice."""
    n_dim = int(np.ceil(N**(1/3)))
    spacing = L_box / n_dim
    positions = []
    for i in range(n_dim):
        for j in range(n_dim):
            for k in range(n_dim):
                if len(positions) < N:
                    positions.append([i * spacing, j * spacing, k * spacing])
    return np.array(positions)


def init_velocities(N, T0, mass):
    """Initialize velocities from Maxwell-Boltzmann distribution, scaled to target temperature."""
    velocities = np.random.randn(N, 3)
    # Remove center-of-mass motion
    velocities -= velocities.mean(axis=0)
    # Scale to desired temperature
    current_T = np.sum(0.5 * mass * np.sum(velocities**2, axis=1)) / (1.5 * N)
    velocities *= np.sqrt(T0 / current_T)
    return velocities

In [ ]:
def compute_forces(positions, N, L_box, eps, sigma_lj, r_cut2):
    """Compute Lennard-Jones forces and potential energy using minimum image convention."""
    forces = np.zeros_like(positions)
    potential_energy = 0.0
    sigma2 = sigma_lj**2

    for i in range(N - 1):
        for j in range(i + 1, N):
            r_vec = positions[i] - positions[j]
            # Minimum image convention
            r_vec -= L_box * np.round(r_vec / L_box)
            r2 = np.dot(r_vec, r_vec)

            if r2 < r_cut2:
                inv_r6 = (sigma2 / r2)**3
                inv_r12 = inv_r6**2
                f_mag = 48.0 * eps / r2 * (inv_r12 - 0.5 * inv_r6)
                force_vec = f_mag * r_vec
                forces[i] += force_vec
                forces[j] -= force_vec
                potential_energy += 4.0 * eps * (inv_r12 - inv_r6)

    return forces, potential_energy

In [ ]:
def run_simulation(N, L_box, dt, n_steps, T0, mass, eps, sigma_lj, r_cut2):
    """Run molecular dynamics using the Velocity Verlet algorithm."""
    positions = init_positions(N, L_box)
    velocities = init_velocities(N, T0, mass)
    forces, pe = compute_forces(positions, N, L_box, eps, sigma_lj, r_cut2)

    # Store trajectory for animation (every 10th step)
    save_every = 10
    n_saved = n_steps // save_every
    trajectory = np.zeros((n_saved, N, 3))
    pe_trace = np.zeros(n_steps)
    idx = 0

    for step in range(n_steps):
        # Velocity Verlet: update positions
        positions += velocities * dt + 0.5 * forces / mass * dt**2
        positions %= L_box  # periodic boundary conditions

        old_forces = forces.copy()
        forces, pe = compute_forces(positions, N, L_box, eps, sigma_lj, r_cut2)

        # Velocity Verlet: update velocities
        velocities += 0.5 * (forces + old_forces) / mass * dt

        pe_trace[step] = pe

        if step % save_every == 0 and idx < n_saved:
            trajectory[idx] = positions.copy()
            idx += 1

        if step % 500 == 0:
            print(f"  Step {step}/{n_steps}, PE = {pe:.2f}")

    return positions, trajectory, pe_trace

In [ ]:
print("Starting simulation...")
final_positions, trajectory, pe_trace = run_simulation(
    N, L_box, dt, n_steps, T0, mass, eps, sigma_lj, r_cut2
)
print("Simulation finished.")

In [ ]:
# 2D projection animation
fig, ax = plt.subplots(figsize=(6, 6))

def update(frame):
    ax.cla()
    ax.scatter(trajectory[frame, :, 0], trajectory[frame, :, 1], s=20)
    ax.set_xlim(0, L_box)
    ax.set_ylim(0, L_box)
    ax.set_aspect("equal")
    ax.set_title(f"Step {frame * 10}")

anim = FuncAnimation(fig, update, frames=range(0, len(trajectory), 2), interval=50)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
def calculate_rdf(positions, N, L_box, rho, n_bins=100):
    """Compute the radial distribution function."""
    dr = L_box / 2 / n_bins
    g = np.zeros(n_bins)

    for i in range(N - 1):
        for j in range(i + 1, N):
            r_vec = positions[i] - positions[j]
            r_vec -= L_box * np.round(r_vec / L_box)
            r = np.linalg.norm(r_vec)
            if r < L_box / 2:
                bin_idx = int(r / dr)
                if bin_idx < n_bins:
                    g[bin_idx] += 2  # count both (i,j) and (j,i)

    # Normalize
    rdf_r = (np.arange(n_bins) + 0.5) * dr
    for i in range(n_bins):
        r_inner = i * dr
        r_outer = (i + 1) * dr
        shell_vol = 4/3 * np.pi * (r_outer**3 - r_inner**3)
        g[i] /= (N * rho * shell_vol)

    return rdf_r, g

In [ ]:
rdf_r, rdf_g = calculate_rdf(final_positions, N, L_box, rho)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rdf_r, rdf_g, linewidth=2)
ax.set_xlabel("r (distance)")
ax.set_ylabel("g(r)")
ax.set_title("Radial Distribution Function")
ax.axhline(1, color="gray", linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.show()